# fedOPD · build runbook — forecaster + federated loop

Successor to the EDA (closed at tag `eda-v2`). Turns the frozen 125-trust panel into trained
federated forecasters with an honest verdict against the **F-27 bar (naive-t-1 ~12.6% sMAPE)**.

**Conventions (see `docs/BUILD_LOG.md`, `docs/OPS.md`):**
- **Notebook-only** build, sectioned by step (owner instruction — overrides the runbook's modules+CLI
  proposal, recorded as D-06). Code here is AI-authored under the owner's explicit override of §12.
- **Device: CPU by default.** CUDA is used only if >20 GB is free (checked read-only so the shared,
  busy A100 is never disturbed). Whole grid ~7 TFLOP = CPU-minutes.
- Steps run in order; each ends in a **GATE**. A red gate STOPS the line.
- Findings are `R-xx`; decisions continue from `D-06`. Every run writes `results/*.json`.

## Step 0 · Preflight
Verify the inherited canonical table (D-01), pick the device without disturbing the GPU, cap threads,
and stamp the environment. **GATE:** row/month/column asserts pass and `results/env.json` is written.

In [1]:
# Step 0 - preflight: env, device policy, inherited canonical table.
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys, json, hashlib, platform, subprocess, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CLAUDE.md").exists())
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)

# Device: use CUDA only if >20 GB is free (checked read-only via nvidia-smi, so we never disturb the
# shared A100), else CPU. The whole grid is ~7 TFLOP -> CPU-minutes, so CPU blocks nothing.
def gpu_free_gb():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5)
        return int(r.stdout.strip().splitlines()[0]) / 1024
    except Exception:
        return None
_free = gpu_free_gb()
DEVICE = "cuda" if (_free is not None and _free > 20 and torch.cuda.is_available()) else "cpu"
print(f"device: {DEVICE}  (gpu free: {None if _free is None else round(_free, 1)} GB)")

# Inherited canonical table (D-01) - the ONLY input; never re-read the raw CSVs.
canon = pd.read_parquet(ROOT / "reports" / "tables" / "rtt_canonical.parquet")
EXPECT_COLS = ["date", "branch", "branch_name", "icb", "dept", "dept_name", "demand"]
assert len(canon) == 227344, f"row count {len(canon)} != 227344 - EDA/build disagree, STOP"
assert canon.date.nunique() == 96, "month count != 96"
assert list(canon.columns) == EXPECT_COLS, f"columns {list(canon.columns)}"
print(f"canonical OK: {len(canon):,} rows | {canon.date.nunique()} months "
      f"{canon.date.min().date()}..{canon.date.max().date()}")

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return "?"

env = {"when": dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "host": platform.node(),
       "git": git_sha(), "python": sys.version.split()[0], "torch": torch.__version__,
       "numpy": np.__version__, "pandas": pd.__version__, "device": DEVICE,
       "gpu_free_gb": None if _free is None else round(_free, 1),
       "cpu_count": os.cpu_count(), "omp_threads": os.environ.get("OMP_NUM_THREADS"),
       "rows": int(len(canon)), "months": int(canon.date.nunique())}
(RESULTS / "env.json").write_text(json.dumps(env, indent=2))
print("GATE Step 0: PASS - asserts pass, env stamped ->", (RESULTS / "env.json"))

device: cpu  (gpu free: 6.6 GB)
canonical OK: 227,344 rows | 96 months 2018-04-01..2026-03-01
GATE Step 0: PASS - asserts pass, env stamped -> /home/garima/fedOPD/results/env.json


## Step 1 · Freeze the panel, roster, and eligible series
Apply D-03 (125-trust panel) and the F-28 roster decision (18 window-complete departments, excluding
the X 'Other' family - **D-07, pending supervisor Q5**). Filter to series that can yield >=24 train
windows and >=1 test window under window W=12 (D-02 validity). Freeze `results/panel.json` (hashed);
everything downstream reads this file. **GATE:** 125 trusts, series+drops logged, panel.json written.

In [2]:
# Step 1 - freeze panel, department roster, and eligible series -> results/panel.json
from collections import Counter
months = pd.Index(sorted(canon.date.unique()))              # 96-month axis
TRAIN_END = pd.Timestamp("2025-03-01")                      # last train target month (D-04)
W = 12                                                       # input window (F-20 persistence, F-22 weak seasonality)
MIN_TRAIN_WINDOWS = 24
DEPTS_MODE = "stable18"                                      # default roster (F-28); alts: all24, no_x
MIN_COVERAGE = 0.5                                          # supervisor Q4 default

# D-03 panel: trusts present in all 96 months (relaxed variants recorded for the supervisor).
pm = canon.groupby("branch").date.nunique()
panel = sorted(pm.index[pm == 96])
variants = {"ge96_default": int((pm == 96).sum()), "ge90": int((pm >= 90).sum()), "ge84": int((pm >= 84).sum())}
assert len(panel) == 125, f"panel {len(panel)} != 125"
pset = set(panel)

# Roster (F-28): departments present in all 96 months = the 18 stable specialties (excludes X01 36mo,
# X02-X06 60mo). D-07 default excludes the catch-all X 'Other' family (pending supervisor Q5).
dept_months = canon.groupby("dept").date.nunique()
stable18 = sorted(dept_months.index[dept_months == 96])
roster_full = {"stable18": stable18,
               "all24": sorted(canon.dept.unique()),
               "no_x": sorted(d for d in canon.dept.unique() if not str(d).startswith("X"))}
roster = roster_full[DEPTS_MODE]

# Coverage filter (Q4): keep depts offered by >= MIN_COVERAGE of the 125 panel trusts.
cov = canon[canon.branch.isin(pset)].groupby("dept").branch.nunique() / len(panel)
roster = [d for d in roster if cov.get(d, 0) >= MIN_COVERAGE]

# Series eligibility (D-02 window validity): a window at target index t is valid iff months[t-W:t] are
# all present AND month t is present. Eligible iff >= MIN_TRAIN_WINDOWS train windows and >= 1 test window.
tr_idx = [i for i, m in enumerate(months) if i >= W and m <= TRAIN_END]
te_idx = [i for i, m in enumerate(months) if m > TRAIN_END]
pres = (canon.assign(one=1).pivot_table("one", ["branch", "dept"], "date", fill_value=0)
        .reindex(columns=months, fill_value=0).astype(bool))

series, drops = [], []
for (b, d), row in pres.iterrows():
    if b not in pset:   drops.append((b, d, "not-in-panel")); continue
    if d not in roster: drops.append((b, d, "dept-not-in-roster")); continue
    v = row.values
    tr = sum(bool(v[i]) and bool(v[i - W:i].all()) for i in tr_idx)
    te = sum(bool(v[i]) and bool(v[i - W:i].all()) for i in te_idx)
    if te < 1:                  drops.append((b, d, "no-test-window")); continue
    if tr < MIN_TRAIN_WINDOWS:  drops.append((b, d, "few-train-windows")); continue
    series.append([b, d, int(tr), int(te)])

reasons = Counter(r for b, d, r in drops if r not in ("not-in-panel", "dept-not-in-roster"))
print(f"panel {len(panel)} trusts | roster {len(roster)} of {len(roster_full[DEPTS_MODE])} depts "
      f"(mode={DEPTS_MODE}, cov>={MIN_COVERAGE})")
print(f"eligible series: {len(series)} | informative drops: {dict(reasons)}")

panel_obj = {"created": env["when"], "git": env["git"], "panel_trusts": panel,
             "panel_variants": variants, "depts_mode": DEPTS_MODE, "roster": roster,
             "min_coverage": MIN_COVERAGE, "window": W, "min_train_windows": MIN_TRAIN_WINDOWS,
             "train_end": str(TRAIN_END.date()), "test_months": [str(months[i].date()) for i in te_idx],
             "series": series, "n_series": len(series), "drop_reasons": dict(reasons)}
panel_obj["hash"] = hashlib.sha256(json.dumps(panel_obj, sort_keys=True).encode()).hexdigest()[:16]
(RESULTS / "panel.json").write_text(json.dumps(panel_obj, indent=2))
gate = (len(panel) == 125 and len(series) > 0 and (RESULTS / "panel.json").exists())
print("panel.json hash:", panel_obj["hash"])
print("GATE Step 1:", "PASS" if gate else "FAIL", "->", (RESULTS / "panel.json"))

panel 125 trusts | roster 17 of 18 depts (mode=stable18, cov>=0.5)
eligible series: 1528 | informative drops: {'no-test-window': 155, 'few-train-windows': 16}
panel.json hash: c106fd0afe4baeb3
GATE Step 1: PASS -> /home/garima/fedOPD/results/panel.json
